Installing scGen via
 
 pip install git+https://github.com/theislab/scgen.git 
 
 and running

 pip install pertpy

 pip install arviz

 pip install toytree

 pip install formulaic

 pip install pydeseq2


should be enough for running this notebook. Python 3.10 was a working version in my experience.

This notebook provides a simple scenario from loading data, splitting based on the defined scenario, creating a perturbation benchmark object and adding benchamrking models to it, training and prediciton, and calculating metrics. 

All classes/modules/functions need to be improved and generalized further, and all ca use this notebook (and improve it if you can) for testing their implementaion. 

In [1]:
import numpy as np
import scanpy as sc
import torch

In [2]:
from dataset import PerturbationDataset
from benchmark import PerturbationBenchmark
from models.scgen import ScGenModel


from scenarios.splits.splits import Scenarios, Scenario1
from scenarios.transforms.method_transform import MethodTransform

/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### End-to-end pipeline for benchmarking perturbation predictions using perturbench

##### 1. Data processing - example provided by scenario and dataset team

In [3]:
# load the data
adata = sc.datasets.moignard15()
adata

/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


AnnData object with n_obs × n_vars = 3934 × 42
    obs: 'exp_groups'
    uns: 'iroot', 'exp_groups_colors'

In [4]:
# Create some dummy perturbation columns
adata.obs['condition'] = np.random.choice(['A', 'B', 'control'], adata.shape[0])

# Create some dummy cell-type columns
adata.obs['cell_type'] = np.random.choice(['alpha', 'beta', 'gamma'], adata.shape[0])
adata

AnnData object with n_obs × n_vars = 3934 × 42
    obs: 'exp_groups', 'condition', 'cell_type'
    uns: 'iroot', 'exp_groups_colors'

In [5]:
# Create a Method Transform object
method_transform = MethodTransform(
    adata=adata,
    gpu=False
)

# Process the data and return the processed anndata object
method_transform.process_data(
    method="scgen",
    celltype_key="cell_type",
    perturbation_key="condition",
)
processed_data=method_transform.return_anndata()

In [6]:
processed_data

AnnData object with n_obs × n_vars = 3934 × 42
    obs: 'exp_groups', 'condition', 'cell_type', 'batch_key', 'labels_key'
    uns: 'iroot', 'exp_groups_colors'

In [7]:
# Create a dummy 'dataset' variable and split the anndata based on this covariate 
dataset = np.random.choice(['norman', 'replogle'], processed_data.shape[0])
adata_train = processed_data[dataset == 'norman']
adata_test = processed_data[dataset == 'replogle']

##### 2. Define scenario

In [8]:
# Read in scenario config
import yaml

scenario_config_path = "scenarios/splits/configs/scenario_1_scgen.yaml"
try:
    with open(scenario_config_path, 'r') as file:
        scenario_config = yaml.safe_load(file)
    print(scenario_config)
except FileNotFoundError:
    print(f"Error: The file '{scenario_config_path}' was not found.")
except yaml.YAMLError as exc:
    print(f"Error parsing YAML file: {exc}")

{'method': 'scgen', 'test_size': 1, 'split_test': False, 'val_size': 0.3, 'scenario_name': 'Scenario 1 scGen', 'train_dataset': 'Train1', 'test_dataset': 'Test1', 'perturbation_key': 'condition', 'cell_type_key': 'cell_type', 'min_cells': 25, 'seed': 0}


In [9]:
scenario_config

{'method': 'scgen',
 'test_size': 1,
 'split_test': False,
 'val_size': 0.3,
 'scenario_name': 'Scenario 1 scGen',
 'train_dataset': 'Train1',
 'test_dataset': 'Test1',
 'perturbation_key': 'condition',
 'cell_type_key': 'cell_type',
 'min_cells': 25,
 'seed': 0}

In [10]:
# Initialize scenario class
scenario = Scenarios(
    scenario=Scenario1,
    adata_train=adata_train,
    adata_test=adata_test,
    scenario_config=scenario_config
)
scenario

In [11]:
# Perform train, test, val split using the splitter function in the Scenario class
# which is wrapped in the 'return_data' method of Scenarios 

# This will return three anndata objects - train val and test
adata_train, adata_val, adata_test = scenario.return_data()

/Users/zeinab/OneDrive - University of Toronto/phd/project/perturbation_benchmark/scEvalsJam/perturbench/scenarios/splits/splits.py:72: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  self.adata_train.obs['split'] = 'train'
/Users/zeinab/OneDrive - University of Toronto/phd/project/perturbation_benchmark/scEvalsJam/perturbench/scenarios/splits/splits.py:73: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  self.adata_test.obs['split'] = 'test'
/Users/zeinab/OneDrive - University of Toronto/phd/project/perturbation_benchmark/scEvalsJam/perturbench/scenarios/splits/splits.py:96: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_train.obs['split'] = train_split_key
/Users/zeinab/OneDrive - University of Toronto/phd/project/perturbation_benchmark/scEvalsJam/perturbench/scenarios/splits/splits.py:97: ImplicitModificationWarnin

##### Standardize datasets

In [12]:
covariate_fields = ["cell_type", "condition", "split", "batch_key", "labels_key", "dataset"]
train_standardized_dataset = PerturbationDataset(
    anndata=adata_train,
    perturbation_field="condition",
    covariate_fields=covariate_fields)

val_standardized_dataset = PerturbationDataset(
    anndata=adata_val,
    perturbation_field="condition",
    covariate_fields=covariate_fields)

test_standardized_dataset = PerturbationDataset(
    anndata=adata_test,
    perturbation_field="condition",
    covariate_fields=covariate_fields)

In [13]:
test_standardized_dataset.anndata()

AnnData object with n_obs × n_vars = 2031 × 42
    obs: 'cell_type', 'condition', 'split', 'batch_key', 'labels_key', 'dataset', 'perturbation'

##### 3. Run model training

initialize scgen model and add it to the list of benchamrking models

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark_model_list = []
scgen_model = ScGenModel(adata_train=train_standardized_dataset.anndata(),
                         conditions_key="condition",
                         cell_label_key="cell_type",
                         labels_key="cell_type",
                         ctrl_key="control",
                         stim_key="A",
                         batch_key="condition",
                         cond_to_predict="A",
                         save_model_path="/saved_models",
                         device=device)
benchmark_model_list.append(scgen_model)
benchmark_model_list

##### 5. Register this all with a benchmark object

Now, let's register this all and check what's going to be run!

In [15]:
benchmark = PerturbationBenchmark()
benchmark.models

[]

In [16]:
# register datasets
benchmark_dataset = {
    "train": train_standardized_dataset,
    "test": test_standardized_dataset,
}
benchmark.add_dataset(benchmark_dataset)
benchmark.train_test_dict

{'train': <dataset.PerturbationDataset at 0x7fcc336b3f70>,
 'test': <dataset.PerturbationDataset at 0x7fcc33782980>}

In [17]:
# register models
for model in benchmark_model_list:
    benchmark.add_model(model)
    
benchmark.models

For now, we consider all default metric of the evaluation class. Ideally the perturbation benchmark class should enable defining desired evaluation metrics for benchmarking.

In [18]:
# register metrics
# for metric in [my_favourite_metric]:
#     benchmark.add_metric(metric)

train models added to the PerturbationBenchmark object

In [19]:
benchmark.train()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 5/5: 100%|██████████| 5/5 [00:03<00:00,  1.62it/s, v_num=1, train_loss_step=194, train_loss_epoch=178]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 5/5: 100%|██████████| 5/5 [00:03<00:00,  1.59it/s, v_num=1, train_loss_step=194, train_loss_epoch=178]
Model scGen training completed successfully


make prediction

In [20]:
benchmark.predict()

self.batch_key:  condition
self.ctrl_key:  control
condition  condition
A          A            677
B          B            692
control    control      662
dtype: int64
train_data:
AnnData object with n_obs × n_vars = 1332 × 42
    obs: 'cell_type', 'condition', 'split', 'batch_key', 'labels_key', 'dataset', 'perturbation', '_scvi_batch', '_scvi_labels'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
before predict:
View of AnnData object with n_obs × n_vars = 662 × 42
    obs: 'cell_type', 'condition', 'split', 'batch_key', 'labels_key', 'dataset', 'perturbation'
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setu

/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/zeinab/opt/anaconda3/envs/perturbenchmark/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/zeinab/opt/an

And finally let's view the results as a table! - this section needs to be completed

In [21]:
# benchmark.calculate_metrics(
#     adata_test=test_standardized_dataset.anndata(),
#     control_label="control",
#     target_pert_list=["A", "B"],
#     condition_label="condition",
#     deg_count=20
# )


type of adata_test.X <class 'anndata._core.views.ArrayView'>


Exception: Input anndata_x.X is not sparse.csr_matrix